In [3]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import requests
import joblib
from datetime import datetime, timedelta
import pytz
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Cell 2: Configurations
LATITUDE = 21.0050
LONGITUDE = 106.8333
MODEL_PATH = 'ensemble_model_pm25.pkl'
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast" # Hoặc dùng archive nếu cần

BASE_MET_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_u', 'wind_v']

# Cell 3: Core Functions
def get_daily_metrics(y_true, y_pred):
    """Tính toán sai số MAE và RMSE"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

# Cell 4: Main Execution
if __name__ == "__main__":
    print("--- QUY TRÌNH TỰ ĐỘNG ĐÁNH GIÁ 30 NGÀY (BẢN FIX LỖI GỌI API) ---")
    
    # 1. Tải mô hình và Scaler
    try:
        model_data = joblib.load(MODEL_PATH)
        model = model_data['model']
        feature_names = model_data['feature_names']
        
        # Nếu scaler được đóng gói trong model_data thì lấy ra, nếu không tải từ file rời
        scaler = model_data.get('scaler') 
        if scaler is None:
            scaler = joblib.load('scaler.pkl')
    except Exception as e:
        print(f"Lỗi tải mô hình: {e}. Vui lòng đảm bảo file mô hình và scaler đang nằm ở cùng thư mục.")
        exit()

    # 2. Xác định khung thời gian 30 ngày
    vn_tz = pytz.timezone('Asia/Bangkok')
    end_date = datetime.now(vn_tz).date() - timedelta(days=1) # Lùi 1 ngày để đảm bảo có đủ 24h thực tế
    start_date = end_date - timedelta(days=29)
    
    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')
    print(f"Đang tải đồng loạt dữ liệu từ {start_str} đến {end_str}...")
    
    try:
        # 3. Gọi API Khí tượng (Nhiệt, Ẩm, Gió, Mưa...)
        w_params = {
            "latitude": LATITUDE, "longitude": LONGITUDE,
            "start_date": start_str, "end_date": end_str,
            "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m",
            "timezone": "Asia/Bangkok"
        }
        w_res = requests.get(WEATHER_API_URL, params=w_params).json()
        if 'hourly' not in w_res:
            w_res = requests.get("https://archive-api.open-meteo.com/v1/archive", params=w_params).json()
        df_weather = pd.DataFrame(w_res['hourly'])
        
        # 4. Gọi API Khí thải (PM2.5, NO2, SO2...)
        aq_params = {
            "latitude": LATITUDE, "longitude": LONGITUDE,
            "start_date": start_str, "end_date": end_str,
            "hourly": "pm2_5,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth",
            "timezone": "Asia/Bangkok"
        }
        aq_res = requests.get(AIR_QUALITY_API_URL, params=aq_params).json()
        df_aq = pd.DataFrame(aq_res['hourly'])
        
        # 5. Hợp nhất hai luồng dữ liệu (Merge)
        df_merged = pd.merge(df_weather, df_aq, on='time').dropna()
        
        # 6. Tiền xử lý dữ liệu đồng loạt
        df_merged['time'] = pd.to_datetime(df_merged['time'])
        df_merged['date'] = df_merged['time'].dt.date
        df_merged['hour'] = df_merged['time'].dt.hour
        df_merged['day_of_week'] = df_merged['time'].dt.dayofweek
        df_merged['month'] = df_merged['time'].dt.month
        
        # Trích xuất vector gió
        rad = np.radians(df_merged['wind_direction_10m'].astype(float))
        df_merged['wind_u'] = -df_merged['wind_speed_10m'] * np.sin(rad)
        df_merged['wind_v'] = -df_merged['wind_speed_10m'] * np.cos(rad)
        
        # Hàm ghi nhớ đặc trưng 24h
        for feature in BASE_MET_FEATURES:
            df_merged[f'{feature}_rolling_12h'] = df_merged[feature].rolling(window=12, min_periods=1).mean()
            df_merged[f'{feature}_rolling_24h'] = df_merged[feature].rolling(window=24, min_periods=1).mean()
            
        # Loại bỏ các dòng đầu tiên không đủ dữ liệu rolling
        df_merged = df_merged.dropna()
        
        # 7. Tính toán cho từng ngày
        final_results = []
        for current_date in df_merged['date'].unique():
            df_day = df_merged[df_merged['date'] == current_date].copy()
            
            # Chỉ đánh giá những ngày có đủ 24 giờ dữ liệu
            if len(df_day) < 24:
                continue 
                
            # Chuẩn bị dữ liệu vào AI
            X = df_day[feature_names]
            X_scaled = scaler.transform(X)
            y_pred = model.predict(X_scaled)
            y_true = df_day['pm2_5'].values
            
            mae, rmse = get_daily_metrics(y_true, y_pred)
            
            final_results.append({
                "Ngày": current_date.strftime('%Y-%m-%d'), 
                "MAE (µg/m³)": round(mae, 2), 
                "RMSE (µg/m³)": round(rmse, 2),
                "PM2.5 Thực tế TB": round(np.mean(y_true), 2)
            })
            print(f"Thành công: {current_date} | MAE: {mae:.2f}")
            
        # 8. Xuất báo cáo
        df_report = pd.DataFrame(final_results)
        if not df_report.empty:
            df_report.to_csv("summary_30days_metrics.csv", index=False, encoding='utf-8-sig')
            print("\n" + "="*60)
            print("BẢNG THỐNG KÊ PHỤC VỤ BÁO CÁO KỶ YẾU")
            print("="*60)
            print(df_report.to_string(index=False))
            print("-" * 60)
            print(f"MAE Trung bình 30 ngày  : {df_report['MAE (µg/m³)'].mean():.2f}")
            print(f"RMSE Trung bình 30 ngày : {df_report['RMSE (µg/m³)'].mean():.2f}")
        else:
            print("Không có đủ dữ liệu để tính toán.")
            
    except Exception as e:
        print(f"Lỗi hệ thống: {e}")

--- QUY TRÌNH TỰ ĐỘNG ĐÁNH GIÁ 30 NGÀY (BẢN FIX LỖI GỌI API) ---
Đang tải đồng loạt dữ liệu từ 2026-03-15 đến 2026-04-13...
Lỗi hệ thống: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- precipitation_rolling_12h
- precipitation_rolling_24h
- relative_humidity_2m_rolling_12h
- relative_humidity_2m_rolling_24h
- temperature_2m_rolling_12h
- ...

